<a href="https://colab.research.google.com/github/Nihat0311/NLP/blob/main/Extractive_question_answering_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!curl -L -o stanford-question-answering-dataset.zip\
  https://www.kaggle.com/api/v1/datasets/download/stanfordu/stanford-question-answering-dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 8937k  100 8937k    0     0  5966k      0  0:00:01  0:00:01 --:--:-- 8937k


In [ ]:
!unzip /content/stanford-question-answering-dataset.zip

Archive:  /content/stanford-question-answering-dataset.zip
replace dev-v1.1.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import json
import pandas as pd

def parse_squad_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        squad_data = json.load(f)

    data_rows = []
    for article in squad_data['data']:
        for paragraph in article['paragraphs']:
            context = paragraph['context']
            for qa in paragraph['qas']:
                question = qa['question']
                id_ = qa['id']
                if 'answers' in qa and len(qa['answers']) > 0:
                    answers = {"text": [ans['text'] for ans in qa['answers']]}
                else:
                    continue

                data_rows.append({
                    "id": id_,
                    "context": context,
                    "question": question,
                    "answers": answers
                })

    return pd.DataFrame(data_rows)

json_file_path = "dev-v1.1.json"

df_all = parse_squad_json(json_file_path)

df = df_all.head(500).copy()
print(f"Dataset uğurla yükləndi! Toplam sətir sayı: {len(df)}")

In [ ]:
df_all.info()

In [ ]:
import matplotlib.pyplot as plt

df["context_len"] = df["context"].apply(lambda x: len(x.split()))
df["question_len"] = df["question"].apply(lambda x: len(x.split()))
df["answer_len"] = df["answers"].apply(lambda x: len(x["text"][0].split()) if len(x["text"]) > 0 else 0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(df["context_len"], bins=30, color="skyblue", edgecolor="black")
axes[0].set_title("Context Length Distribution")
axes[1].hist(df["question_len"], bins=20, color="lightgreen", edgecolor="black")
axes[1].set_title("Question Length Distribution")
axes[2].hist(df["answer_len"], bins=15, color="salmon", edgecolor="black")
axes[2].set_title("Answer Length Distribution")
plt.tight_layout()
plt.show()

In [ ]:
import time
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def load_qa_model(model_name):
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device)
    return tokenizer, model

distilbert_name = "distilbert-base-uncased-distilled-squad"
bert_large_name = "bert-large-uncased-whole-word-masking-finetuned-squad"

distil_tokenizer, distil_model = load_qa_model(distilbert_name)
bert_tokenizer, bert_model = load_qa_model(bert_large_name)

In [ ]:
def get_answer(question, context, tokenizer, model):
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
    answer_start = torch.argmax(outputs.start_logits)
    answer_end = torch.argmax(outputs.end_logits) + 1

    answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][answer_start:answer_end]))
    return answer.strip()

context_ex = "Albert Einstein was born in Germany in 1879. He later developed the theory of relativity."
question_ex = "Where was Albert Einstein born?"

print("DistilBERT Answer:", get_answer(question_ex, context_ex, distil_tokenizer, distil_model))

In [ ]:
import collections
import re
import string


def normalize_answer(s):
    """Mətni kiçik hərflərə çevirir, punktuasiyanı və artiklları silir."""
    def remove_articles(text): return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text): return ' '.join(text.split())
    def remove_punc(text): return ''.join(ch for ch in text if ch not in set(string.punctuation))
    def lower(text): return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def compute_em(prediction, truth):
    return int(normalize_answer(prediction) == normalize_answer(truth))

def compute_f1(prediction, truth):
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(truth).split()
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)

    common = collections.Counter(pred_tokens) & collections.Counter(truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0

    precision = 1.0 * num_same / len(pred_tokens)
    recall = 1.0 * num_same / len(truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1

In [ ]:
eval_dataset = df_all.head(100).to_dict(orient="records")

def evaluate_model(tokenizer, model, dataset):
    em_scores = []
    f1_scores = []
    import time
    from tqdm import tqdm
    start_time = time.time()
    analysis_data = []

    for item in tqdm(dataset, desc="Evaluating"):
        context = item["context"]
        question = item["question"]
        truths = item["answers"]["text"]

        pred = get_answer(question, context, tokenizer, model)

        if "[cls]" in pred.lower() or pred == "":
            pred = ""

        em = max([compute_em(pred, t) for t in truths]) if truths else 0
        f1 = max([compute_f1(pred, t) for t in truths]) if truths else 0

        em_scores.append(em)
        f1_scores.append(f1)

        analysis_data.append({
            "context_len": len(context.split()),
            "em": em,
            "f1": f1,
            "pred": pred,
            "truth": truths[0] if truths else ""
        })

    total_time = time.time() - start_time
    avg_em = (sum(em_scores) / len(em_scores)) * 100
    avg_f1 = (sum(f1_scores) / len(f1_scores)) * 100
    avg_speed = total_time / len(dataset)

    return avg_em, avg_f1, avg_speed, pd.DataFrame(analysis_data)
distil_em, distil_f1, distil_speed, distil_df = evaluate_model(distil_tokenizer, distil_model, eval_dataset)
bert_em, bert_f1, bert_speed, bert_df = evaluate_model(bert_tokenizer, bert_model, eval_dataset)

distil_size = sum(p.numel() for p in distil_model.parameters()) / 1e6
bert_size = sum(p.numel() for p in bert_model.parameters()) / 1e6

In [ ]:
comparison_table = pd.DataFrame({
    "Metric": ["Exact Match (EM %)", "F1-Score (%)", "Avg Inference Time (sec)", "Model Size (M params)"],
    "DistilBERT": [distil_em, distil_f1, distil_speed, distil_size],
    "BERT-Large": [bert_em, bert_f1, bert_speed, bert_size]
})

print("\n=== MODEL COMPARISON ===")
print(comparison_table.to_markdown(index=False))

In [ ]:
def analyze_by_length(df_analysis, name):
    print(f"\n--- {name} Length Analysis ---")
    short_ctx = df_analysis[df_analysis["context_len"] < 150]
    long_ctx = df_analysis[df_analysis["context_len"] >= 150]

    print(f"Qısa Context (<150 söz) -> F1: {short_ctx['f1'].mean()*100:.2f}%, Sayı: {len(short_ctx)}")
    print(f"Uzun Context (>=150 söz) -> F1: {long_ctx['f1'].mean()*100:.2f}%, Sayı: {len(long_ctx)}")

    errors = df_analysis[df_analysis["em"] == 0].head(2)
    print(f"\nSəhv edilən nümunələrə misal ({name}):")
    for idx, row in errors.iterrows():
        print(f"Həqiqi Cavab: {row['truth']} | Modelin Cavabı: {row['pred']}")

analyze_by_length(distil_df, "DistilBERT")
analyze_by_length(bert_df, "BERT-Large")

In [ ]:
context_exp = "Albert Einstein was born in Germany in 1879. He later developed the theory of relativity."
questions = [
    "Where was Einstein born?",
    "What country was Einstein born in?",
    "Einstein's birthplace?"
]

print("\n=== Robustness Test ===")
for q in questions:
    ans_distil = get_answer(q, context_exp, distil_tokenizer, distil_model)
    ans_bert = get_answer(q, context_exp, bert_tokenizer, bert_model)
    print(f"Question: '{q}'")
    print(f"  -> DistilBERT: {ans_distil}")
    print(f"  -> BERT-Large: {ans_bert}\n")

In [ ]:
context_exp = "Hesen is very good boy . He is 20 years old . Hesen is an AI engineer"
questions = [
    "What is Hesen's gender?",
    "What is Hesen's age?",
    "What is Hesen's profession?"
]

print("\n=== Robustness Test ===")
for q in questions:
    ans_distil = get_answer(q, context_exp, distil_tokenizer, distil_model)
    ans_bert = get_answer(q, context_exp, bert_tokenizer, bert_model)
    print(f"Question: '{q}'")
    print(f"  -> DistilBERT: {ans_distil}")
    print(f"  -> BERT-Large: {ans_bert}\n")

In [ ]:
context_exp = "If you are familiar with Object-Oriented Programming, then you’ve probably heard about the SOLID principles.These five software development principles are guidelines to follow when building software so that it is easier to scale and maintain. They were made popular by a software engineer, Robert C. Martin.There are so many great articles online about SOLID but I rarely see any examples with pictures. This makes it a bit difficult for visual learners like me to learn while staying engaged.So the main aim of this article is to get a better understanding of these principles using illustrations and emphasizing the goal for each principle.You see, some of these principles may look similar but they are not targeting the same goal. It is possible to satisfy one principle while violating the other, even though they are alike.To make this as simple to follow, I will be using the word “Class” but note that it can also apply to a Function, Method or Module in this article.Update*I got some comments about the Open Closed in this article violating the Single Responsibility Principle. Please note that the aim of this article is to explain each of these principles independent of the others.Also, Responsibilities(or Roles) are different from Actions. In SRP, I used “I am Painter”, in Open-Closed, I used “I can Paint”.It is important to note this because several actions can be performed to fulfil a responsibility(or role). The class should have one responsibility(SRP) but its functionality that fulfils that responsibility should be open to extend(OCP).Now, let’s begin!The SOLID Principles.S — Single Responsibility.A class should have a single responsibilityIf a Class has many responsibilities, it increases the possibility of bugs because making changes to one of its responsibilities, could affect the other ones without you knowing.Goal:This principle aims to separate behaviours so that if bugs arise as a result of your change, it won’t affect other unrelated behaviours.O — Open-ClosedClasses should be open for extension, but closed for modificationChanging the current behaviour of a Class will affect all the systems using that Class.If you want the Class to perform more functions, the ideal approach is to add to the functions that already exist NOT change them.GoalThis principle aims to extend a Class’s behaviour without changing the existing behaviour of that Class. This is to avoid causing bugs wherever the Class is being used.L — Liskov SubstitutionIf S is a subtype of T, then objects of type T in a program may be replaced with objects of type S without altering any of the desirable properties of that program.When a child Class cannot perform the same actions as its parent Class, this can cause bugs.If you have a Class and create another Class from it, it becomes a parent and the new Class becomes a child. The child Class should be able to do everything the parent Class can do. This process is called Inheritance.The child Class should be able to process the same requests and deliver the same result as the parent Class or it could deliver a result that is of the same type.The picture shows that the parent Class delivers Coffee(it could be any type of coffee). It is acceptable for the child Class to deliver Cappucino because it is a specific type of Coffee, but it is NOT acceptable to deliver Water.If the child Class doesn’t meet these requirements, it means the child Class is changed completely and violates this principle.GoalThis principle aims to enforce consistency so that the parent Class or its child Class can be used in the same way without any errors.I — Interface SegregationClients should not be forced to depend on methods that they do not use.When a Class is required to perform actions that are not useful, it is wasteful and may produce unexpected bugs if the Class does not have the ability to perform those actions.A Class should perform only actions that are needed to fulfil its role. Any other action should be removed completely or moved somewhere else if it might be used by another Class in the future.GoalThis principle aims at splitting a set of actions into smaller sets so that a Class executes ONLY the set of actions it requires.D — Dependency Inversion- High-level modules should not depend on low-level modules. Both should depend on the abstraction.- Abstractions should not depend on details. Details should depend on abstractions.Firstly, let’s define the terms used here more simplyHigh-level Module(or Class): Class that executes an action with a tool.Low-level Module (or Class): The tool that is needed to execute the actionAbstraction: Represents an interface that connects the two Classes.Details: How the tool worksThis principle says a Class should not be fused with the tool it uses to execute an action. Rather, it should be fused to the interface that will allow the tool to connect to the Class.It also says that both the Class and the interface should not know how the tool works. However, the tool needs to meet the specification of the interface.GoalThis principle aims at reducing the dependency of a high-level Class on the low-level Class by introducing an interface.SummarySo far, we have discussed these five principles and highlighted their goals. They are to help you make your code easy to adjust, extend and test with little to no problems.Thank you so much for reading. I hope you have a better idea about this topic and you had as much fun reading this as I did writing it.Soli principles can be used in Java programing language.It is favorite principles for programmers"
questions = [
    "How many prinsiples in SOLID?",
    "What is S in SOLID?",
    "What is D in SOLID?"
    ]

print("\n=== Robustness Test ===")
for q in questions:
    ans_distil = get_answer(q, context_exp, distil_tokenizer, distil_model)
    ans_bert = get_answer(q, context_exp, bert_tokenizer, bert_model)
    print(f"Question: '{q}'")
    print(f"  -> DistilBERT: {ans_distil}")
    print(f"  -> BERT-Large: {ans_bert}\n")

In [ ]:
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def create_knowledge_base(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        squad_data = json.load(f)

    unique_contexts = set()
    for article in squad_data['data']:
        for paragraph in article['paragraphs']:
            unique_contexts.add(paragraph['context'])

    return list(unique_contexts)

json_file_path = "dev-v1.1.json"
knowledge_base = create_knowledge_base(json_file_path)
print(f"Bazada toplam {len(knowledge_base)} ədəd unikal kontekst (mətn) var.")

In [ ]:
def add_context(new_context, knowledge_base):
    if new_context not in knowledge_base:
        knowledge_base.append(new_context)

In [ ]:
add_context(new)

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
context_vectors = vectorizer.fit_transform(knowledge_base)

def find_best_context(question, knowledge_base, context_vectors):
    question_vector = vectorizer.transform([question])
    similarities = cosine_similarity(question_vector, context_vectors).flatten()
    best_match_idx = np.argmax(similarities)

    return knowledge_base[best_match_idx], similarities[best_match_idx]

In [ ]:
def ask_the_system(question, tokenizer, model):
    retrieved_context, score = find_best_context(question, knowledge_base, context_vectors)

    print(f"\n[Sistem]: Sualınız üçün bazadan uyğun kontekst tapıldı (Bənzərlik skoru: {score:.2f})")
    print(f"[Tapılan Kontekst Snippet]: {retrieved_context}...")
    answer = get_answer(question, retrieved_context, tokenizer, model)

    print(f"[Modelin Cavabı]: {answer}")
    return answer
user_question = "What team did the Panthers defeat?"
ask_the_system(user_question, bert_tokenizer, bert_model)


[Sistem]: Sualınız üçün bazadan uyğun kontekst tapıldı (Bənzərlik skoru: 0.18)
[Tapılan Kontekst Snippet]: The Panthers defense gave up just 308 points, ranking sixth in the league, while also leading the NFL in interceptions with 24 and boasting four Pro Bowl selections. Pro Bowl defensive tackle Kawann Short led the team in sacks with 11, while also forcing three fumbles and recovering two. Fellow lineman Mario Addison added 6½ sacks. The Panthers line also featured veteran defensive end Jared Allen, a 5-time pro bowler who was the NFL's active career sack leader with 136, along with defensive end Kony Ealy, who had 5 sacks in just 9 starts. Behind them, two of the Panthers three starting linebackers were also selected to play in the Pro Bowl: Thomas Davis and Luke Kuechly. Davis compiled 5½ sacks, four forced fumbles, and four interceptions, while Kuechly led the team in tackles (118) forced two fumbles, and intercepted four passes of his own. Carolina's secondary featured Pro Bowl

'carolina'